In [1]:
import os
import pandas as pd
import bev
import cv2
import numpy as np
from tqdm import tqdm
import json
import time
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [3]:
# Constants

NUM_ROWS = 8192

DATASET_PATH = "../../../dataset"
IMAGES_A_PATH = os.path.join(DATASET_PATH, 'images_A')
IMAGES_B_PATH = os.path.join(DATASET_PATH, 'images_B')
METADATA_PATH = os.path.join(DATASET_PATH, 'metadata_A')

DATAFRAME_PATH = "../../eval"
SAVE_PATH = os.path.join(DATAFRAME_PATH, "pipeline_results.csv")

MODEL_PATH = "../../models/pose_landmarker_full.task"

FOCAL_LENGTH_MM = 35.0
SENSOR_WIDTH_MM = 36.0
IMAGE_HEIGHT_PX = 1080
IMAGE_WIDTH_PX = 1920
BASELINE_M = 0.1

In [4]:
# Load Dataframe

df = pd.read_csv(SAVE_PATH, dtype={'id': str})
df.head(6)

,id,actor,pose,cam_pitch,gt_dist,gt_ang_sep,gt_d_yaw,gt_d_pitch,gt_sw_x,gt_sw_y,...,m2st_dist,m2st_ang_sep,m2st_d_yaw,m2st_d_pitch,m2st_sw_x,m2st_sw_y,m2st_sw_z,m2st_sc_x,m2st_sc_y,m2st_sc_z
0,00001,BP_Ada_C_1,34.0,-6.287881,354.167297,14.391177,-4.068234,13.838668,-0.968621,0.066612,...,337.681453,160.060166,-172.649296,7.383960,0.935750,0.127283,0.328893,-0.999921,-0.002682,-0.012263
1,00002,BP_Ada_C_1,17.0,-80.392527,839.159817,70.426628,10.291927,-70.265796,-0.335014,-0.175880,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00003,BP_Ada_C_1,114.0,-50.585528,716.956094,30.322529,-20.436334,29.535145,-0.863197,0.059908,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00004,BP_Ada_C_1,42.0,-20.623707,357.440408,61.804887,-68.575333,9.502579,-0.472476,0.805154,...,386.860524,16.545744,-17.400630,3.438862,-0.960390,0.272383,0.058802,-0.999872,-0.002752,-0.015739
4,00005,BP_Ada_C_1,114.0,-16.400097,540.643029,72.647749,82.989366,63.720576,-0.298245,-0.170291,...,483.475293,165.921985,175.796565,-18.033845,0.967381,-0.071326,0.243077,-0.999933,-0.001848,-0.011442
5,00006,BP_Ada_C_1,92.0,-67.072543,511.871710,35.985887,-115.477473,3.051334,-0.809162,0.306925,...,452.631735,175.191351,-172.912755,-127.839991,0.994123,0.070675,0.081999,-0.999674,-0.011433,-0.022845


In [5]:
# Configure BEV Settings

settings = bev.main.default_settings
settings.mode = 'image'
settings.render_mesh = False  # Skip the 3D mesh creation
settings.show = False         # Skip opening a display window
settings.save_dict = False    # Skip saving .npz files to disk

In [6]:
# Load BEV Model

bev_model = bev.BEV(settings)

Using BEV.
Threshold for positive center detection: 0.08


c:\Users\ExoHorizon\miniconda3\envs\fbv-bev\lib\site-packages\bev\main.py:101: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(self.settings.m

In [7]:
# Initialize SGBM object

stereo = cv2.StereoSGBM_create(
    minDisparity=0,
    numDisparities=160,
    blockSize=5,
    P1=8 * 3 * 5**2,
    P2=32 * 3 * 5**2,
    disp12MaxDiff=1,
    uniquenessRatio=15,
    speckleWindowSize=0,
    speckleRange=2,
    preFilterCap=63,
    mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY
)

In [8]:
def run_model_pipeline(sample_id, camera_pitch_rad):

    # Load Image
    image_A_path = F"{IMAGES_A_PATH}/{sample_id}A.png"
    image_B_path = F"{IMAGES_B_PATH}/{sample_id}B.png"

    image = cv2.imread(image_A_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    image_A = cv2.imread(image_A_path)
    image_B = cv2.imread(image_B_path)

    gray_A_full = cv2.cvtColor(image_A, cv2.COLOR_BGR2GRAY)
    gray_B_full = cv2.cvtColor(image_B, cv2.COLOR_BGR2GRAY)

    start_time = time.perf_counter()
    # ------------------------------------

    # Crop Images
    h, w = 480, 480
    y_start, x_start = 300, 720

    gray_A = gray_A_full[y_start:y_start+h, x_start:x_start+w]
    gray_B = gray_B_full[y_start:y_start+h, x_start:x_start+w]

    # Run Inference
    outputs = bev_model(image)
    joints = outputs['joints']
    if hasattr(joints, 'detach'):
        joints = joints.detach().cpu().numpy()

    if joints is None or joints.size == 0:
        return [np.nan] * 10, 0

    # Get 2D Keypoints
    def get_normalized_2d_keypoints(outputs, person_idx=0):
        cam = outputs['cam'][person_idx]
        if hasattr(cam, 'detach'):
            cam = cam.detach().cpu().numpy()
            
        scale = cam[0]
        trans = cam[1:3]

        joints_3d = outputs['joints'][person_idx]
        if hasattr(joints_3d, 'detach'):
            joints_3d = joints_3d.detach().cpu().numpy()
        
        kp2d_ndc = scale * (joints_3d[:, :2] + trans)

        keypoints_2d_norm = (kp2d_ndc + 1.0) / 2.0
        
        return keypoints_2d_norm

    keypoints_2d = get_normalized_2d_keypoints(outputs)

    # Create Numpy Arrays
    keypoints_3d = joints[0].copy()

    # Compute Disparity
    disparity = stereo.compute(gray_A, gray_B).astype(np.float32) / 16.0

    # Get Predicted Right Shoulder Pixel Location
    height, width = disparity.shape

    shoulder_kp_2d = keypoints_2d[17]

    px_x = int(np.clip(shoulder_kp_2d[0] * width, 0, width - 1))
    px_y = int(np.clip(shoulder_kp_2d[1] * height, 0, height - 1))

    window = disparity[px_y-2:px_y+3, px_x-2:px_x+3]
    disp_at_shoulder = np.nanmedian(window) if np.any(window) else 0

    # Calculate Focal Length
    focal_length_px = (FOCAL_LENGTH_MM * IMAGE_WIDTH_PX) / SENSOR_WIDTH_MM

    # Distance Output
    distance = None
    if disp_at_shoulder > 0:
        distance = (focal_length_px * BASELINE_M) / disp_at_shoulder
    else:
        return [np.nan] * 10, 0
    
    # Transform 3D Keypoints Given Distance
    keypoints_3d_rel = keypoints_3d - keypoints_3d[17]
    transformed_keypoints_3d = keypoints_3d_rel.copy()
    transformed_keypoints_3d[:, 2] += distance

    # ------------------------------------
    inference_time = time.perf_counter() - start_time
    
    # Apply Coordinate Conversion
    def convert_to_unreal_coords(points_3d_meters):
        unreal_pts = np.zeros_like(points_3d_meters)
        unreal_pts[:, 0] = points_3d_meters[:, 2] * 100
        unreal_pts[:, 1] = points_3d_meters[:, 0] * 100
        unreal_pts[:, 2] = -points_3d_meters[:, 1] * 100
        return unreal_pts

    ue_keypoints_3d = convert_to_unreal_coords(transformed_keypoints_3d)

    # Given Constants
    camera_coords = np.array([0, 0, 0])
    world_up = np.array([0, 0, 1])

    # Get Right Shoulder & Wrist Coordinates
    shoulder_coords = ue_keypoints_3d[17]
    wrist_coords = ue_keypoints_3d[21]

    # Shoulder-Camera Distance
    distance = np.linalg.norm(shoulder_coords)

    # Shoulder-Wrist Shoulder-Camera Vectors
    shoulder_wrist = wrist_coords - shoulder_coords
    shoulder_wrist /= np.linalg.norm(shoulder_wrist)

    shoulder_camera = camera_coords - shoulder_coords
    shoulder_camera /= np.linalg.norm(shoulder_camera)

    # Angular Separation
    angular_separation_rad = np.arccos(np.clip(np.dot(shoulder_wrist, shoulder_camera), -1.0, 1.0))
    angular_separation_deg = np.rad2deg(angular_separation_rad)

    # Camera Un-Pitch Matrix (Aligns Z-Axis with Gravity)
    c, s = np.cos(-camera_pitch_rad), np.sin(-camera_pitch_rad)
    unpitch_matrix = np.array([
        [c,  0, s],
        [0,  1, 0],
        [-s, 0, c]
    ])

    # Un-Pitch Vectors
    shoulder_wrist_gravity = unpitch_matrix @ shoulder_wrist
    shoulder_wrist_gravity /= np.linalg.norm(shoulder_wrist_gravity)

    shoulder_camera_gravity = unpitch_matrix @ shoulder_camera
    shoulder_camera_gravity /= np.linalg.norm(shoulder_camera_gravity)

    # Yaw & Pitch Components
    shoulder_wrist_gravity_yaw = np.rad2deg(np.atan2(shoulder_wrist_gravity[1], shoulder_wrist_gravity[0]))
    shoulder_wrist_gravity_pitch = np.rad2deg(np.asin(np.clip(shoulder_wrist_gravity[-1], -1.0, 1.0)))

    shoulder_camera_gravity_yaw = np.rad2deg(np.atan2(shoulder_camera_gravity[1], shoulder_camera_gravity[0]))
    shoulder_camera_gravity_pitch = np.rad2deg(np.asin(np.clip(shoulder_camera_gravity[-1], -1.0, 1.0)))

    delta_yaw = (shoulder_wrist_gravity_yaw - shoulder_camera_gravity_yaw + 180) % 360 - 180
    delta_pitch = shoulder_wrist_gravity_pitch - shoulder_camera_gravity_pitch

    # Relevant Outputs
    return distance, angular_separation_deg, delta_yaw, delta_pitch, \
           shoulder_wrist[0], shoulder_wrist[1], shoulder_wrist[2], \
           shoulder_camera[0], shoulder_camera[1], shoulder_camera[2], \
           inference_time

In [9]:
# Execution Loop

active_pipeline = 'b3sd' 
pipeline_cols = [f'{active_pipeline}_{m}' for m in ['dist', 'ang_sep', 'd_yaw', 'd_pitch', 'sw_x', 'sw_y', 'sw_z', 'sc_x', 'sc_y', 'sc_z']]

total_inference_seconds = 0.0
successful_counts = 0

print(f"Running Inference for Pipeline: {active_pipeline}")

for i in tqdm(range(len(df)), desc=f"Processing {active_pipeline}"):
    row_id = df.at[i, 'id']
    
    pitch_deg = df.at[i, 'cam_pitch']
    pitch_rad = np.deg2rad(pitch_deg)
    
    try:
        *results, elapsed = run_model_pipeline(row_id, pitch_rad)
        
        df.loc[i, pipeline_cols] = results
        
        total_inference_seconds += elapsed
        successful_counts += 1

    except Exception as e:
        # tqdm.write(f"Pipeline {active_pipeline} failed for ID {row_id}: {e}")
        continue

print(f"\nTotal Cumulative Inference Time: {total_inference_seconds:.4f} seconds")
if successful_counts > 0:
    print(f"Average per sample: {(total_inference_seconds / successful_counts) * 1000:.2f} ms")

Running Inference for Pipeline: b3sd


Processing b3sd:   4%|▍         | 337/8192 [01:08<24:57,  5.25it/s]

No person detected!


Processing b3sd:   5%|▍         | 408/8192 [01:23<24:35,  5.28it/s]

No person detected!


...

No person detected!


Processing b3sd: 100%|█████████▉| 8153/8192 [29:53<00:08,  4.74it/s]

No person detected!


Processing b3sd: 100%|██████████| 8192/8192 [30:02<00:00,  4.54it/s]


Total Cumulative Inference Time: 634.3957 seconds
Average per sample: 86.63 ms


In [10]:
print(df.info(verbose=True, show_counts=True))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8192 entries, 0 to 8191
Data columns (total 84 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            8192 non-null   object 
 1   actor         8192 non-null   object 
 2   pose          8192 non-null   float64
 3   cam_pitch     8192 non-null   float64
 4   gt_dist       8192 non-null   float64
 5   gt_ang_sep    8192 non-null   float64
 6   gt_d_yaw      8192 non-null   float64
 7   gt_d_pitch    8192 non-null   float64
 8   gt_sw_x       8192 non-null   float64
 9   gt_sw_y       8192 non-null   float64
 10  gt_sw_z       8192 non-null   float64
 11  gt_sc_x       8192 non-null   float64
 12  gt_sc_y       8192 non-null   float64
 13  gt_sc_z       8192 non-null   float64
 14  b3ad_dist     0 non-null      float64
 15  b3ad_ang_sep  0 non-null      float64
 16  b3ad_d_yaw    0 non-null      float64
 17  b3ad_d_pitch  0 non-null      float64
 18  b3ad_sw_x     0 non-null    

In [11]:
# Save Dataframe

df.to_csv(SAVE_PATH, index=False)

In [12]:
# Sanity Check

check_df = pd.read_csv(SAVE_PATH, dtype={'id': str})
print(check_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8192 entries, 0 to 8191
Data columns (total 84 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            8192 non-null   object 
 1   actor         8192 non-null   object 
 2   pose          8192 non-null   float64
 3   cam_pitch     8192 non-null   float64
 4   gt_dist       8192 non-null   float64
 5   gt_ang_sep    8192 non-null   float64
 6   gt_d_yaw      8192 non-null   float64
 7   gt_d_pitch    8192 non-null   float64
 8   gt_sw_x       8192 non-null   float64
 9   gt_sw_y       8192 non-null   float64
 10  gt_sw_z       8192 non-null   float64
 11  gt_sc_x       8192 non-null   float64
 12  gt_sc_y       8192 non-null   float64
 13  gt_sc_z       8192 non-null   float64
 14  b3ad_dist     0 non-null      float64
 15  b3ad_ang_sep  0 non-null      float64
 16  b3ad_d_yaw    0 non-null      float64
 17  b3ad_d_pitch  0 non-null      float64
 18  b3ad_sw_x     0 non-null    